# Pipeline de Análisis de Neurofeedback (Muse 2)
Este cuaderno importa automáticamente los datos del repositorio de GitHub y genera gráficas comparativas de las sesiones.

In [ ]:
# 1. Clonar el repositorio y moverse a la carpeta
!git clone https://github.com/Laroguee/neurofeedback-muse.git
%cd neurofeedback-muse

In [ ]:
import os
import glob
import numpy as np
import json
import matplotlib.pyplot as plt
import scipy.signal as signal

# 2. Buscar automáticamente todas las sesiones en la carpeta "data"
rutas_npy = sorted(glob.glob("data/*_raw.npy"))
sesiones = {}

for ruta in rutas_npy:
    nombre_sesion = os.path.basename(ruta).replace("_raw.npy", "")
    data = np.load(ruta)
    with open(f"data/{nombre_sesion}_meta.json", "r") as f:
        meta = json.load(f)
    sesiones[nombre_sesion] = {"data": data, "meta": meta}

print(f"✅ Se encontraron y cargaron {len(sesiones)} sesiones: {list(sesiones.keys())}")

In [ ]:
# 3. Gráfica Comparativa de Espectro (PSD) para ver la evolución
plt.figure(figsize=(12, 6))

for nombre_sesion, info in sesiones.items():
    datos = info["data"]
    meta = info["meta"]
    sfreq = meta["sampling_rate"]
    
    # Elegimos un canal frontal (AF7), índice 1
    canal_idx = meta["eeg_channel_indices"][1] 
    nombre_canal = meta["eeg_channel_names"][1]
    senial = datos[canal_idx]
    
    freqs, psd = signal.welch(senial, sfreq, nperseg=int(sfreq*2))
    freq_mask = (freqs >= 1) & (freqs <= 40)
    
    plt.plot(freqs[freq_mask], psd[freq_mask], label=f"{nombre_sesion}", linewidth=2)

plt.title(f'Evolución del Paciente: Comparativa de Sesiones (Canal {nombre_canal})', fontsize=15)
plt.xlabel('Frecuencia (Hz) [1-40 Hz]', fontsize=12)
plt.ylabel('Poder de la Onda (dB/Hz)', fontsize=12)

plt.axvspan(4, 8, color='blue', alpha=0.1, label='Theta (Somnolencia)')
plt.axvspan(8, 12, color='orange', alpha=0.15, label='Alfa (Relajación)')
plt.axvspan(12, 30, color='green', alpha=0.1, label='Beta (Concentración)')

plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()